# 🎯 Arabic CV Analyzer
## AraBERT vs CAMeL-BERT | Attention Pooling + Fixed Thresholds + Domain ATS

| Section | Description |
|---------|-------------|
| 1 | Imports & Config |
| 2 | Data Exploration |
| 3 | ATS Engine + Preprocessing |
| 4 | Score → Class Mapping |
| 5 | Model Architecture |
| 6 | Training |
| 7 | Evaluation |
| 8 | Explainability & Inference |
| 9 | 💾 Save Everything for API |

---
## 1️⃣ Imports & Config

In [ ]:
# !pip install transformers torch scikit-learn pandas numpy matplotlib seaborn -q

In [ ]:
import re, warnings, json, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    mean_absolute_error, mean_squared_error, r2_score
)
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

# ── Paths ─────────────────────────────────────────────
SAVE_DIR = Path('saved_model')   # كل الـ artifacts هتتحفظ هنا
SAVE_DIR.mkdir(exist_ok=True)

# ── Device ────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Device: {device}')

# ── Model registry ────────────────────────────────────
MODELS = {
    'AraBERT'   : 'aubmindlab/bert-base-arabertv2',
    'CAMeL-BERT': 'CAMeL-Lab/bert-base-arabic-camelbert-mix'
}

CLASS_NAMES  = ['ضعيف', 'متوسط', 'جيد', 'ممتاز']
CLASS_LABELS = [0, 1, 2, 3]

# ── Fixed hyperparams (بدل Grid Search) ───────────────
BEST_PARAMS = {
    'lr'        : 2e-5,
    'batch_size': 8,
    'dropout'   : 0.3,
}
FULL_EPOCHS = 5

print(f'📌 Hyperparams: {BEST_PARAMS}')
print(f'📁 Save dir   : {SAVE_DIR.resolve()}')

---
## 2️⃣ Data Exploration

In [ ]:
df = pd.read_csv('arabic_cvs_with_scores.csv', encoding='utf-8-sig')
df['cv_words'] = df['arabic_cv'].astype(str).apply(lambda x: len(x.split()))

print(f'✅ {len(df)} CVs | Columns: {list(df.columns)}')
df.head(3)

In [ ]:
print(df.info())
print('\nMissing:', df.isnull().sum().to_dict())
print('\n', df[['suitability_score']].describe())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Data Exploration', fontsize=16, fontweight='bold')

cc = df['category'].value_counts()
axes[0,0].barh(cc.index[:12], cc.values[:12], color='steelblue')
axes[0,0].set_title('CV Count per Category')

axes[0,1].hist(df['suitability_score'].dropna(), bins=20,
               color='royalblue', edgecolor='white')
axes[0,1].axvline(df['suitability_score'].mean(), color='red',
                   linestyle='--', label=f"Mean={df['suitability_score'].mean():.1f}")
axes[0,1].set_title('Suitability Score Distribution')
axes[0,1].legend()

axes[1,0].hist(df['cv_words'], bins=30, color='coral', edgecolor='white')
axes[1,0].axvline(df['cv_words'].mean(), color='navy', linestyle='--',
                   label=f"Mean={df['cv_words'].mean():.0f}")
axes[1,0].set_title('CV Word Count')
axes[1,0].legend()

cs = df.groupby('category')['suitability_score'].mean().sort_values()
axes[1,1].barh(cs.index, cs.values, color='mediumpurple')
axes[1,1].set_title('Avg Suitability per Category')

plt.tight_layout()
plt.savefig('01_exploration.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3️⃣ ATS Engine + Preprocessing

In [ ]:
# ── Domain Keywords ───────────────────────────────────
DOMAIN_KEYWORDS = {
    'تكنولوجيا المعلومات': [
        'python','java','javascript','sql','nosql','react','node','docker',
        'kubernetes','aws','azure','git','api','rest','machine learning',
        'deep learning','linux','agile','scrum','ci/cd'
    ],
    'موارد بشرية': [
        'توظيف','تدريب','تطوير','أداء','رواتب','تعويضات','علاقات عمل',
        'kpi','okr','onboarding','hris','payroll','retention','culture',
        'succession','talent','engagement'
    ],
    'مالية ومحاسبة': [
        'excel','sap','oracle','ifrs','gaap','ميزانية','تدقيق','ضرائب',
        'تحليل مالي','تقارير','quickbooks','erp','cpa','cma','dcf'
    ],
    'هندسة': [
        'autocad','solidworks','matlab','ansys','pmp','iso','six sigma',
        'lean','revit','primavera','bim','حديد','خرسانة','ميكانيكا',
        'كهرباء','plc','scada'
    ],
    'تسويق': [
        'seo','sem','google ads','facebook ads','content','analytics',
        'crm','hubspot','mailchimp','brand','roi','kpi','digital',
        'social media','copywriting','strategy'
    ],
    'مبيعات': [
        'مبيعات','عملاء','crm','salesforce','target','pipeline',
        'negotiation','b2b','b2c','revenue','quota','cold calling',
        'account management','upsell'
    ],
    'رعاية صحية': [
        'سريري','مرضى','تشخيص','علاج','أدوية','مختبر','تمريض',
        'طوارئ','جراحة','emr','his','bls','acls','hipaa'
    ],
    'تعليم': [
        'مناهج','تدريس','طلاب','تقييم','تعليم إلكتروني','lms',
        'moodle','خطة درس','أهداف تعليمية','bloom','differentiation'
    ],
    'إعلام وتصميم': [
        'photoshop','illustrator','figma','adobe','premiere','after effects',
        'typography','branding','ui','ux','indesign','3d','animation'
    ],
    'سياحة ومطاعم': [
        'hospitality','opera','خدمة عملاء','haccp','food safety',
        'فنادق','مطاعم','حجوزات','pos','revenue management'
    ],
}
DEFAULT_KEYWORDS = [
    'خبرة','مهارات','تواصل','قيادة','تحليل','مشاريع','فريق',
    'إدارة','تطوير','تحسين','excel','word','powerpoint'
]

ATS_MESSAGES = {
    'has_experience': 'أضف قسم خبرة العمل بالتفصيل',
    'has_education' : 'أضف قسم التعليم والمؤهلات',
    'has_skills'    : 'أضف قسم المهارات التقنية والشخصية',
    'has_summary'   : 'أضف ملخصاً مهنياً في بداية الـ CV',
    'has_email'     : 'أضف بريدك الإلكتروني',
    'has_phone'     : 'أضف رقم هاتفك',
    'good_length'   : 'اجعل الـ CV بين 200-800 كلمة',
    'has_contact'   : 'أضف رابط LinkedIn أو GitHub أو Portfolio'
}

print('✅ Domain keywords & ATS config loaded')

In [ ]:
def calculate_ats_score(arabic_cv, category=''):
    cv = str(arabic_cv).lower()
    checks = {
        'has_experience': any(k in cv for k in ['الخبرات','خبرة العمل','الخبرة المهنية']),
        'has_education' : any(k in cv for k in ['التعليم','المؤهلات','الدراسة']),
        'has_skills'    : any(k in cv for k in ['المهارات','الكفاءات','قدرات']),
        'has_summary'   : any(k in cv for k in ['ملخص','نبذة','عن نفسي','profile']),
        'has_email'     : bool(re.search(r'[\w\.-]+@[\w\.-]+', cv)),
        'has_phone'     : bool(re.search(r'[\+\d][\d\s\-]{8,}', cv)),
        'good_length'   : 200 <= len(cv.split()) <= 800,
        'has_contact'   : any(k in cv for k in ['linkedin','github','portfolio','موقع'])
    }
    struct_weights = {
        'has_experience':15,'has_education':12,'has_skills':12,'has_summary':8,
        'has_email':6,'has_phone':4,'good_length':6,'has_contact':7
    }
    struct_score  = sum(struct_weights[k] for k, v in checks.items() if v)
    keywords      = DOMAIN_KEYWORDS.get(category, DEFAULT_KEYWORDS)
    matched       = [k for k in keywords if k.lower() in cv]
    keyword_score = (len(matched) / max(len(keywords), 1)) * 30
    total         = min(int(struct_score + keyword_score), 100)
    return total, checks, matched


def clean_arabic_text(text):
    text = str(text)
    text = re.sub(r'[\u064B-\u065F]', '', text)
    text = re.sub(r'[إأآا]', 'ا', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'[^\w\s\u0600-\u06FF]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


# Apply ATS
ats_r              = df.apply(lambda r: calculate_ats_score(r['arabic_cv'], r['category']), axis=1)
df['ats_score']    = ats_r.apply(lambda x: x[0])
df['ats_checks']   = ats_r.apply(lambda x: x[1])
df['ats_matched']  = ats_r.apply(lambda x: x[2])
df['ats_kw_count'] = df['ats_matched'].apply(len)

# Clean + split
df['arabic_cv_clean'] = df['arabic_cv'].apply(clean_arabic_text)
df['input_text']      = df['category'] + ' [SEP] ' + df['arabic_cv_clean']
df_clean = df.dropna(subset=['arabic_cv_clean', 'suitability_score']).copy()

train_df, temp_df = train_test_split(df_clean, test_size=.2, random_state=42)
val_df,   test_df = train_test_split(temp_df,  test_size=.5, random_state=42)

print(f'✅ ATS applied | Train:{len(train_df)} Val:{len(val_df)} Test:{len(test_df)}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Domain-specific ATS Analysis', fontweight='bold')

checks_df   = pd.DataFrame(df['ats_checks'].tolist())
checks_mean = checks_df.mean().sort_values()
axes[0].barh(checks_mean.index, checks_mean.values*100,
             color=['seagreen' if v>.7 else 'coral' for v in checks_mean.values])
axes[0].axvline(70, color='red', linestyle='--', alpha=.5, label='70% threshold')
axes[0].set_title('ATS Structural Checks Pass Rate (%)')
axes[0].legend()
for i, v in enumerate(checks_mean.values):
    axes[0].text(v*100+.5, i, f'{v*100:.0f}%', va='center', fontsize=8)

axes[1].hist(df['ats_kw_count'], bins=20, color='mediumpurple', edgecolor='white')
axes[1].axvline(df['ats_kw_count'].mean(), color='red', linestyle='--',
                label=f"Mean={df['ats_kw_count'].mean():.1f}")
axes[1].set_title('Domain Keyword Match Distribution')
axes[1].legend()

cat_ats = df.groupby('category')['ats_score'].mean().sort_values()
axes[2].barh(cat_ats.index, cat_ats.values,
             color=['seagreen' if v>60 else 'coral' for v in cat_ats.values])
axes[2].axvline(60, color='red', linestyle='--', alpha=.5)
axes[2].set_title('Avg ATS Score per Category')

plt.tight_layout()
plt.savefig('02_ats_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4️⃣ Score → Class Mapping

In [ ]:
T1, T2, T3 = 50, 70, 85

def score_to_class(score):
    if   score < T1: return 0
    elif score < T2: return 1
    elif score < T3: return 2
    else:            return 3

print(f'📌 Thresholds: <{T1}=ضعيف  <{T2}=متوسط  <{T3}=جيد  ≥{T3}=ممتاز')

# Distribution check
df_clean['label'] = df_clean['suitability_score'].apply(score_to_class)
print('\nClass distribution:')
print(df_clean['label'].map(dict(enumerate(CLASS_NAMES))).value_counts())

---
## 5️⃣ Model Architecture

In [ ]:
class CVDatasetSliding(Dataset):
    def __init__(self, df, tokenizer, max_len=512, stride=384):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.stride    = stride

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        toks = self.tokenizer(
            row['input_text'], truncation=True,
            max_length=512,    add_special_tokens=True
        )
        ids             = toks['input_ids']
        chunks_ids, chunks_mask = [], []
        start = 0

        while start < len(ids):
            end   = min(start + self.max_len, len(ids))
            chunk = torch.tensor(ids[start:end], dtype=torch.long)
            mask  = torch.ones(len(chunk),       dtype=torch.long)
            pad   = self.max_len - len(chunk)
            if pad > 0:
                chunk = torch.cat([chunk, torch.zeros(pad, dtype=torch.long)])
                mask  = torch.cat([mask,  torch.zeros(pad, dtype=torch.long)])
            chunks_ids.append(chunk)
            chunks_mask.append(mask)
            if end == len(ids): break
            start += self.stride

        return {
            'chunks_ids' : torch.stack(chunks_ids),
            'chunks_mask': torch.stack(chunks_mask),
            'score'      : torch.tensor(row['suitability_score'] / 100., dtype=torch.float),
            'n_chunks'   : len(chunks_ids)
        }


def collate_fn(batch):
    max_c = max(b['n_chunks'] for b in batch)
    ids_l, mask_l, scores = [], [], []
    for b in batch:
        p = max_c - b['n_chunks']
        if p > 0:
            ids_l.append(torch.cat([b['chunks_ids'],  torch.zeros(p, 512, dtype=torch.long)]))
            mask_l.append(torch.cat([b['chunks_mask'], torch.zeros(p, 512, dtype=torch.long)]))
        else:
            ids_l.append(b['chunks_ids'])
            mask_l.append(b['chunks_mask'])
        scores.append(b['score'])
    return {
        'chunks_ids' : torch.stack(ids_l),
        'chunks_mask': torch.stack(mask_l),
        'n_chunks'   : [b['n_chunks'] for b in batch],
        'score'      : torch.stack(scores)
    }


class AttentionPooling(nn.Module):
    def __init__(self, hidden_size=768):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.Tanh(),
            nn.Linear(256, 1)
        )

    def forward(self, cls_embeddings, n_chunks):
        pooled = []
        for i in range(cls_embeddings.shape[0]):
            n   = n_chunks[i]
            emb = cls_embeddings[i, :n, :]             # (n, 768)
            att = F.softmax(self.attention(emb), dim=0) # (n, 1)
            pooled.append((att * emb).sum(dim=0))       # (768,)
        return torch.stack(pooled)                      # (B, 768)


class CVRegressor(nn.Module):
    def __init__(self, model_path, dropout=0.3):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(model_path)
        self.att_pool = AttentionPooling(hidden_size=768)
        self.head     = nn.Sequential(
            nn.Linear(768, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 64),  nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1),    nn.Sigmoid()
        )

    def forward(self, chunks_ids, chunks_mask, n_chunks):
        B, C, L = chunks_ids.shape
        out  = self.bert(
            input_ids      = chunks_ids.view(B*C, L),
            attention_mask = chunks_mask.view(B*C, L),
            output_attentions = False   # سريع في التدريب
        )
        cls    = out.last_hidden_state[:, 0, :].view(B, C, 768)
        pooled = self.att_pool(cls, n_chunks)
        return self.head(pooled).squeeze(-1)  # (B,)


def run_epoch(model, loader, optimizer, criterion, train=True):
    model.train() if train else model.eval()
    total_loss, preds_list, scores_list = 0, [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            ids      = batch['chunks_ids'].to(device)
            mask     = batch['chunks_mask'].to(device)
            scores   = batch['score'].to(device)
            n_chunks = batch['n_chunks']

            if train: optimizer.zero_grad()
            pred = model(ids, mask, n_chunks)
            loss = criterion(pred, scores)
            if train:
                loss.backward()
                optimizer.step()

            total_loss   += loss.item()
            preds_list.extend((pred * 100).detach().cpu().tolist())
            scores_list.extend((scores * 100).cpu().tolist())

    mae      = mean_absolute_error(scores_list, preds_list)
    rmse     = np.sqrt(mean_squared_error(scores_list, preds_list))
    pred_cls = [score_to_class(p) for p in preds_list]
    true_cls = [score_to_class(s) for s in scores_list]
    acc      = sum(p==t for p,t in zip(pred_cls, true_cls)) / len(true_cls)

    return {
        'loss': total_loss/len(loader),
        'mae' : mae, 'rmse': rmse, 'acc': acc,
        'pred_scores': preds_list, 'true_scores': scores_list,
        'pred_classes': pred_cls,  'true_classes': true_cls
    }


print('✅ Architecture defined')

---
## 6️⃣ Training — AraBERT vs CAMeL-BERT

In [ ]:
final_models  = {}
final_history = {}
tokenizers    = {}

for model_name, model_path in MODELS.items():
    print(f'\n{"="*55}\n🚀 Training: {model_name}\n{"="*55}')

    tok = AutoTokenizer.from_pretrained(model_path)
    tokenizers[model_name] = tok

    train_ds = CVDatasetSliding(train_df, tok)
    val_ds   = CVDatasetSliding(val_df,   tok)
    train_dl = DataLoader(train_ds, batch_size=BEST_PARAMS['batch_size'],
                          shuffle=True, collate_fn=collate_fn)
    val_dl   = DataLoader(val_ds,   batch_size=BEST_PARAMS['batch_size'],
                          collate_fn=collate_fn)

    model     = CVRegressor(model_path, dropout=BEST_PARAMS['dropout']).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=BEST_PARAMS['lr'])
    criterion = nn.MSELoss()

    history  = []
    best_mae = float('inf')
    save_path = SAVE_DIR / f'{model_name.replace("-","_")}_best.pt'

    for epoch in range(FULL_EPOCHS):
        tr = run_epoch(model, train_dl, optimizer, criterion, train=True)
        vl = run_epoch(model, val_dl,   optimizer, criterion, train=False)
        history.append({
            't_loss': tr['loss'], 'v_loss': vl['loss'],
            't_mae' : tr['mae'],  'v_mae' : vl['mae'],
            't_acc' : tr['acc'],  'v_acc' : vl['acc']
        })
        print(f'  Ep{epoch+1}/{FULL_EPOCHS}  '
              f'train_loss={tr["loss"]:.4f}  val_loss={vl["loss"]:.4f}  '
              f'val_mae={vl["mae"]:.2f}  val_acc={vl["acc"]:.3f}')

        if vl['mae'] < best_mae:
            best_mae = vl['mae']
            torch.save({
                'model_state': model.state_dict(),
                'model_name' : model_name,
                'model_path' : model_path,
                'params'     : BEST_PARAMS,
                'thresholds' : (T1, T2, T3),
            }, save_path)
            print(f'  💾 Saved → {save_path}  (val_mae={best_mae:.2f})')

    final_models[model_name]  = model
    final_history[model_name] = history

print('\n✅ Training complete!')

In [ ]:
clrs = {'AraBERT': 'royalblue', 'CAMeL-BERT': 'darkorange'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training Curves — AraBERT vs CAMeL-BERT', fontweight='bold')

for ax, metric, ylabel in zip(axes, ['v_loss','v_mae'], ['Val Loss','Val MAE']):
    for mn, hist in final_history.items():
        ax.plot([h[metric]                for h in hist], '-o',  label=f'{mn} val',   color=clrs[mn])
        ax.plot([h[metric.replace('v_','t_')] for h in hist], '--', label=f'{mn} train', color=clrs[mn], alpha=.4)
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel); ax.set_title(ylabel); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('03_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7️⃣ Evaluation

In [ ]:
eval_results = {}
mnames       = list(MODELS.keys())

for model_name, model_path in MODELS.items():
    save_path  = SAVE_DIR / f'{model_name.replace("-","_")}_best.pt'
    checkpoint = torch.load(save_path, map_location=device)

    # نحمّل الـ best weights
    model = CVRegressor(model_path, dropout=BEST_PARAMS['dropout']).to(device)
    model.load_state_dict(checkpoint['model_state'])
    final_models[model_name] = model

    test_ds = CVDatasetSliding(test_df, tokenizers[model_name])
    test_dl = DataLoader(test_ds, batch_size=8, collate_fn=collate_fn)

    res                      = run_epoch(model, test_dl, None, nn.MSELoss(), train=False)
    res['r2']                = r2_score(res['true_scores'], res['pred_scores'])
    res['spearman']          = spearmanr(res['true_scores'], res['pred_scores'])[0]
    eval_results[model_name] = res

    print(f'\n{"="*55}\n📊 {model_name}')
    print(f'  MAE:{res["mae"]:.2f}  RMSE:{res["rmse"]:.2f}  '
          f'R²:{res["r2"]:.4f}  Spearman:{res["spearman"]:.4f}')
    print(f'\n  Classification Report (Thresholds {T1}/{T2}/{T3}):')
    print(classification_report(res['true_classes'], res['pred_classes'],
                                 target_names=CLASS_NAMES, labels=CLASS_LABELS))

winner = min(eval_results, key=lambda x: eval_results[x]['mae'])
print(f'\n🏆 Winner: {winner}  '
      f'MAE={eval_results[winner]["mae"]:.2f}  '
      f'Acc={eval_results[winner]["acc"]:.3f}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Evaluation — AraBERT vs CAMeL-BERT | Attention Pooling',
             fontsize=13, fontweight='bold')

# 1. Regression metrics
metrics_list = ['mae','rmse','r2','spearman']
x = range(len(metrics_list))
for i, mn in enumerate(mnames):
    vals = [eval_results[mn][m] for m in metrics_list]
    axes[0,0].bar([xi+i*.35 for xi in x], vals, .35, label=mn, color=clrs[mn])
axes[0,0].set_xticks([xi+.175 for xi in x])
axes[0,0].set_xticklabels(['MAE','RMSE','R²','Spearman'])
axes[0,0].set_title('Regression Metrics', fontweight='bold'); axes[0,0].legend()

# 2 & 3. Confusion matrices
for ax_idx, (mn, res) in enumerate(eval_results.items()):
    ax = axes[0, 1] if ax_idx == 0 else axes[0, 2]
    cm = confusion_matrix(res['true_classes'], res['pred_classes'], labels=CLASS_LABELS)
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    title = f'{mn}\nMAE={res["mae"]:.2f}  Acc={res["acc"]:.3f}'
    if mn == winner: title += '  🏆'
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')

# 4. Actual vs Predicted
for mn in mnames:
    axes[1,0].scatter(eval_results[mn]['true_scores'], eval_results[mn]['pred_scores'],
                      alpha=.3, s=8, label=mn, color=clrs[mn])
axes[1,0].plot([0,100],[0,100],'k--', label='Perfect')
for t in [T1, T2, T3]:
    axes[1,0].axhline(t, color='gray', linestyle=':', alpha=.4)
    axes[1,0].axvline(t, color='gray', linestyle=':', alpha=.4)
axes[1,0].set_title('Actual vs Predicted', fontweight='bold'); axes[1,0].legend()

# 5. F1 per class
x2 = range(len(CLASS_NAMES))
for i, mn in enumerate(mnames):
    f1s = f1_score(eval_results[mn]['true_classes'], eval_results[mn]['pred_classes'],
                   average=None, labels=CLASS_LABELS, zero_division=0)
    axes[1,1].bar([xi+i*.35 for xi in x2], f1s, .35, label=mn, color=clrs[mn])
axes[1,1].set_xticks([xi+.175 for xi in x2])
axes[1,1].set_xticklabels(CLASS_NAMES)
axes[1,1].set_title('F1-Score per Class', fontweight='bold'); axes[1,1].legend()

# 6. Error distribution
for mn in mnames:
    errs = np.array(eval_results[mn]['pred_scores']) - np.array(eval_results[mn]['true_scores'])
    axes[1,2].hist(errs, bins=30, alpha=.5,
                   label=f'{mn} (MAE={eval_results[mn]["mae"]:.2f})',
                   color=clrs[mn], edgecolor='white')
axes[1,2].axvline(0, color='red', linestyle='--')
axes[1,2].set_title('Prediction Error Distribution', fontweight='bold'); axes[1,2].legend()

plt.tight_layout()
plt.savefig('04_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8️⃣ Explainability & Inference

In [ ]:
def get_word_importance(text, model, tokenizer, top_k=15):
    """output_attentions=True بس هنا مش في التدريب"""
    model.eval()
    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
    ids = enc['input_ids'].to(device)
    msk = enc['attention_mask'].to(device)

    with torch.no_grad():
        out     = model.bert(input_ids=ids, attention_mask=msk,
                             output_attentions=True)          # ✅ هنا بس
        cls_att = out.attentions[-1][0][:, 0, :].mean(0).cpu().numpy()

    tokens = tokenizer.convert_ids_to_tokens(ids[0].cpu().tolist())
    words, scores = [], []
    cw, cs = '', 0
    for tok, s in zip(tokens[1:-1], cls_att[1:-1]):
        if tok.startswith('##'):
            cw += tok[2:]; cs = max(cs, s)
        else:
            if cw: words.append(cw); scores.append(cs)
            cw, cs = tok, s
    if cw: words.append(cw); scores.append(cs)
    pairs = sorted(zip(words, scores), key=lambda x: -x[1])[:top_k]
    return zip(*pairs) if pairs else ([], [])


SUGGESTION_TEMPLATES = {
    'ضعيف' : [
        (lambda cv,kws: len(kws)<3,            'مهارات المجال ناقصة → أضف ≥5 مهارات تقنية'),
        (lambda cv,kws: 'الخبرات' not in cv,   'لا يوجد قسم خبرة → أضف خبراتك بالتفصيل'),
        (lambda cv,kws: 'المهارات' not in cv,  'قسم المهارات غائب → أضف مهاراتك التقنية والشخصية'),
        (lambda cv,kws: len(cv.split())<150,    'CV قصير جداً → أثرِ المحتوى ليصل 300-500 كلمة'),
        (lambda cv,kws: True,                   'استخدم إنجازات ملموسة بدلاً من المسؤوليات فقط'),
    ],
    'متوسط': [
        (lambda cv,kws: len(kws)<5,                                         'أضف مهارات مجال محددة لرفع الـ ATS score'),
        (lambda cv,kws: 'github' not in cv.lower() and 'linkedin' not in cv.lower(), 'أضف رابط GitHub أو LinkedIn'),
        (lambda cv,kws: 'شهادة' not in cv,                                 'أضف شهادات احترافية معتمدة'),
        (lambda cv,kws: True,                                               'رتّب خبراتك عكسياً زمنياً'),
        (lambda cv,kws: True,                                               'أضف أرقاماً لقياس الإنجازات'),
    ],
    'جيد'  : [
        (lambda cv,kws: len(kws)<8,          'زد الكلمات المفتاحية التقنية لتحسين ATS'),
        (lambda cv,kws: 'مشروع' not in cv,  'أضف قسم مشاريع يوضح أبرز أعمالك'),
        (lambda cv,kws: True,               'خصّص الـ CV لكل وظيفة باستخدام كلمات الإعلان'),
    ],
    'ممتاز': [
        (lambda cv,kws: True, 'CV ممتاز! راجع التنسيق للتوافق مع أنظمة ATS'),
        (lambda cv,kws: True, 'فكر في إضافة قسم توصيات مهنية'),
    ]
}


def analyze_cv(arabic_cv, category, model_name=None):
    if model_name is None:
        model_name = min(eval_results, key=lambda x: eval_results[x]['mae'])

    model      = final_models[model_name]
    tokenizer  = tokenizers[model_name]
    cv_clean   = clean_arabic_text(arabic_cv)
    input_text = f'{category} [SEP] {cv_clean}'

    ds   = CVDatasetSliding(
        pd.DataFrame([{'input_text': input_text,
                       'suitability_score': 0,
                       'category': category}]),
        tokenizer
    )
    item = ds[0]
    model.eval()
    with torch.no_grad():
        pred = model(
            item['chunks_ids'].unsqueeze(0).to(device),
            item['chunks_mask'].unsqueeze(0).to(device),
            [item['n_chunks']]
        )
    pred_score = float(pred.cpu().item()) * 100
    pred_class = score_to_class(pred_score)
    class_name = CLASS_NAMES[pred_class]

    ats, checks, matched_kws = calculate_ats_score(arabic_cv, category)
    domain_kws  = DOMAIN_KEYWORDS.get(category, DEFAULT_KEYWORDS)
    missing_kws = [k for k in domain_kws if k not in matched_kws][:5]

    suggestions = []
    for cond_fn, template in SUGGESTION_TEMPLATES.get(class_name, []):
        if cond_fn(arabic_cv, matched_kws):
            suggestions.append(f'📌 [{class_name}] {template}')
    for check, passed in checks.items():
        if not passed:
            suggestions.append(f'⚠️  ATS: {ATS_MESSAGES[check]}')

    words, wscores = get_word_importance(input_text, model, tokenizer, top_k=10)

    return {
        'model'           : model_name,
        'predicted_score' : round(pred_score, 1),
        'predicted_class' : class_name,
        'thresholds'      : f'{T1}/{T2}/{T3}',
        'ats_score'       : ats,
        'matched_keywords': matched_kws,
        'missing_keywords': missing_kws,
        'suggestions'     : suggestions[:7],
        'top_words'       : list(zip(words, [round(float(s), 4) for s in wscores]))
    }


# ── Sample ────────────────────────────────────────────
sample = test_df.iloc[2]
result = analyze_cv(sample['arabic_cv'], sample['category'])

print(f'Category        : {sample["category"]}')
print(f'True Score      : {sample["suitability_score"]:.1f} → {CLASS_NAMES[score_to_class(sample["suitability_score"])]}')
print(f'Predicted Score : {result["predicted_score"]} → {result["predicted_class"]}')
print(f'ATS Score       : {result["ats_score"]}/100')
print(f'Matched KWs     : {result["matched_keywords"]}')
print(f'Missing KWs     : {result["missing_keywords"]}')
print('\n💡 Suggestions:')
for s in result['suggestions']: print(f'  {s}')
print('\n🔍 Top Words:')
for w, s in result['top_words'][:5]: print(f'  {w}: {s}')

---
## 9️⃣ 💾 Save Everything for API

كل الـ artifacts اللي الـ API هيحتاجها:

| File | Description |
|------|-------------|
| `saved_model/{model}_best.pt` | Model weights + metadata |
| `saved_model/{model}_tokenizer/` | Tokenizer files |
| `saved_model/config.json` | Thresholds + class names + winner |
| `saved_model/ats_config.json` | Domain keywords + ATS messages |
| `saved_model/eval_results.pkl` | Evaluation metrics |
| `saved_model/model_classes.py` | Model class definitions (للـ API يعمل import) |

In [ ]:
# ── 1. Tokenizers ─────────────────────────────────────
for model_name, tok in tokenizers.items():
    tok_path = SAVE_DIR / f'{model_name.replace("-","_")}_tokenizer'
    tok.save_pretrained(tok_path)
    print(f'✅ Tokenizer saved → {tok_path}')

# ── 2. Global config (thresholds / classes / winner) ──
config = {
    'thresholds'  : {'T1': T1, 'T2': T2, 'T3': T3},
    'class_names' : CLASS_NAMES,
    'class_labels': CLASS_LABELS,
    'winner_model': winner,
    'models'      : MODELS,
    'hyperparams' : {**BEST_PARAMS, 'lr': str(BEST_PARAMS['lr'])},  # JSON-safe
    'eval_summary': {
        mn: {'mae': round(r['mae'], 4),
             'rmse': round(r['rmse'], 4),
             'r2': round(r['r2'], 4),
             'spearman': round(r['spearman'], 4),
             'acc': round(r['acc'], 4)}
        for mn, r in eval_results.items()
    }
}
with open(SAVE_DIR / 'config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
print(f'✅ config.json saved')

# ── 3. ATS config (keywords + messages) ───────────────
ats_config = {
    'domain_keywords' : DOMAIN_KEYWORDS,
    'default_keywords': DEFAULT_KEYWORDS,
    'ats_messages'    : ATS_MESSAGES,
}
with open(SAVE_DIR / 'ats_config.json', 'w', encoding='utf-8') as f:
    json.dump(ats_config, f, ensure_ascii=False, indent=2)
print(f'✅ ats_config.json saved')

# ── 4. Full eval results (pickle — arrays included) ───
with open(SAVE_DIR / 'eval_results.pkl', 'wb') as f:
    pickle.dump(eval_results, f)
print(f'✅ eval_results.pkl saved')

print(f'\n📁 All artifacts saved to: {SAVE_DIR.resolve()}')

In [ ]:
# ── 5. model_classes.py — الـ API يعمل import منه ─────
model_classes_code = '''
"""Model architecture — import this in your API"""
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel


class AttentionPooling(nn.Module):
    def __init__(self, hidden_size=768):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.Tanh(),
            nn.Linear(256, 1)
        )

    def forward(self, cls_embeddings, n_chunks):
        pooled = []
        for i in range(cls_embeddings.shape[0]):
            n   = n_chunks[i]
            emb = cls_embeddings[i, :n, :]
            att = F.softmax(self.attention(emb), dim=0)
            pooled.append((att * emb).sum(dim=0))
        return torch.stack(pooled)


class CVRegressor(nn.Module):
    def __init__(self, model_path, dropout=0.3):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(model_path)
        self.att_pool = AttentionPooling(hidden_size=768)
        self.head     = nn.Sequential(
            nn.Linear(768, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 64),  nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1),    nn.Sigmoid()
        )

    def forward(self, chunks_ids, chunks_mask, n_chunks):
        B, C, L = chunks_ids.shape
        out = self.bert(
            input_ids=chunks_ids.view(B*C, L),
            attention_mask=chunks_mask.view(B*C, L),
            output_attentions=False
        )
        cls    = out.last_hidden_state[:, 0, :].view(B, C, 768)
        pooled = self.att_pool(cls, n_chunks)
        return self.head(pooled).squeeze(-1)
'''

with open(SAVE_DIR / 'model_classes.py', 'w', encoding='utf-8') as f:
    f.write(model_classes_code)
print('✅ model_classes.py saved')

# ── Final summary ─────────────────────────────────────
print('\n' + '='*55)
print('📦 saved_model/ contents:')
for p in sorted(SAVE_DIR.rglob('*')):
    if p.is_file():
        size = p.stat().st_size / 1024
        unit = 'KB' if size < 1024 else 'MB'
        size = size if size < 1024 else size/1024
        print(f'  {str(p.relative_to(SAVE_DIR)):<45} {size:.1f} {unit}')

print(f'\n🎉 Done! Winner: {winner} | '
      f'MAE={eval_results[winner]["mae"]:.2f} | '
      f'Acc={eval_results[winner]["acc"]:.3f}')